# DLinear Training 


What it does:
- converts Walmart panel data to Nixtla long format: `unique_id`, `ds`, `y`, and exogenous columns;
- trains one global DLinear model across all Store-Dept series;
- evaluates with the shared WMAE metric on 39-week expanding windows;
- logs the required MLflow stages;
- sketches a pyfunc wrapper so the final artifact can accept raw test rows.

This notebook depends on the deep-learning environment from `requirements-dl.txt`. Do not execute until that environment is active.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

warnings.filterwarnings("ignore")
DATA_DIR = ROOT / "data"
EXPERIMENT_NAME = "DLinear_Training"
MODEL_ALIAS = "DLinear"
HORIZON = 39
RANDOM_STATE = 42

In [ ]:
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

from neuralforecast import NeuralForecast
from neuralforecast.models import DLinear

mlflow.set_experiment(EXPERIMENT_NAME)

# Run this cell once before the stage cells when you want MLflow to group
# the required Cleaning -> Feature Selection -> CV -> HPO -> Final runs.
parent_run = mlflow.start_run(run_name="DLinear_Workflow")

## Data Preparation

`neuralforecast` expects one row per series and date. `unique_id` identifies a Store-Dept time series. The fallback functions below can be replaced by Gega's shared `nf_prep.py` when it is available.

In [ ]:
train_raw, test_raw, features_df, stores_df, sample_submission = load_raw(DATA_DIR)
train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

def to_nixtla(raw_frame: pd.DataFrame, history_frame: pd.DataFrame | None = None, include_y: bool = True) -> pd.DataFrame:
    merged = merge_all(raw_frame, features_df, stores_df)
    history_merged = None if history_frame is None else merge_all(history_frame, features_df, stores_df)
    X = build_features(merged, sales_history_df=history_merged, encode_categoricals=True)
    nf = pd.DataFrame({
        "unique_id": raw_frame["Store"].astype(str) + "_" + raw_frame["Dept"].astype(str),
        "ds": pd.to_datetime(raw_frame["Date"]),
    })
    if include_y and "Weekly_Sales" in raw_frame.columns:
        nf["y"] = raw_frame["Weekly_Sales"].astype(float).to_numpy()
    for col in X.columns:
        values = X[col]
        if str(values.dtype) == "category":
            values = values.astype(str)
        nf[col] = values.to_numpy()
    return nf

def holiday_lookup(raw_frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "unique_id": raw_frame["Store"].astype(str) + "_" + raw_frame["Dept"].astype(str),
        "ds": pd.to_datetime(raw_frame["Date"]),
        "IsHoliday_eval": raw_frame["IsHoliday"].astype(bool).to_numpy(),
    })

## MLflow Stage: Cleaning

Deep models are more sensitive to scale than tree models. The first version keeps negative sales in training but clips final predictions at zero. Sparse or very short series are kept initially; if training fails, filter short series in this stage and log the rule.

In [ ]:
with mlflow.start_run(run_name="DLinear_Cleaning", nested=True):
    series_lengths = train_raw.groupby(["Store", "Dept"]).size()
    mlflow.log_param("negative_sales_training_policy", "keep")
    mlflow.log_param("negative_prediction_policy", "clip_to_zero_for_submission")
    mlflow.log_param("short_series_policy", "keep_initially")
    mlflow.log_metric("series_count", int(series_lengths.shape[0]))
    mlflow.log_metric("min_series_length", int(series_lengths.min()))
    mlflow.log_metric("median_series_length", float(series_lengths.median()))

## MLflow Stage: Feature Selection

DLinear's core strength is decomposing each series into trend and seasonality, so the first feature set is conservative: calendar, holiday, store, markdown, external variables, and safe history features from `features.py`.

In [ ]:
nf_train_preview = to_nixtla(train_raw)
BASE_COLS = ["unique_id", "ds", "y"]
FUTR_EXOG_LIST = [c for c in nf_train_preview.columns if c not in BASE_COLS]

with mlflow.start_run(run_name="DLinear_Feature_Selection", nested=True):
    mlflow.log_param("feature_source", "shared_features_py_encoded_for_neuralforecast")
    mlflow.log_param("uses_future_exog", True)
    mlflow.log_metric("future_exog_count", len(FUTR_EXOG_LIST))
    pd.Series(FUTR_EXOG_LIST, name="feature").to_csv(ROOT / "models" / "dlinear_exog_features.csv", index=False)
    mlflow.log_artifact(str(ROOT / "models" / "dlinear_exog_features.csv"))

## Cross-Validation Helpers

Each fold trains a global DLinear model on all series up to the cutoff and predicts the next 39 dates. WMAE is computed after merging predictions back to the validation rows.

In [ ]:
def build_dlinear(input_size: int = 104, max_steps: int = 500):
    return DLinear(
        h=HORIZON,
        input_size=input_size,
        max_steps=max_steps,
        learning_rate=1e-3,
        futr_exog_list=FUTR_EXOG_LIST,
        alias=MODEL_ALIAS,
        random_seed=RANDOM_STATE,
    )

def cv_dlinear(input_size: int = 104, max_steps: int = 500) -> list[float]:
    scores = []
    for fold, (tr_mask, val_mask) in enumerate(expanding_window_folds(train_raw["Date"], n_splits=3, horizon=HORIZON), start=1):
        fold_train_raw = train_raw.loc[tr_mask].copy()
        fold_val_raw = train_raw.loc[val_mask].copy()
        nf_train = to_nixtla(fold_train_raw)
        nf_val = to_nixtla(fold_val_raw, history_frame=fold_train_raw)
        nf = NeuralForecast(models=[build_dlinear(input_size=input_size, max_steps=max_steps)], freq="W-FRI")
        nf.fit(df=nf_train)
        preds = nf.predict(futr_df=nf_val.drop(columns=["y"]))
        scored = nf_val[["unique_id", "ds", "y"]].merge(preds[["unique_id", "ds", MODEL_ALIAS]], on=["unique_id", "ds"], how="left")
        scored = scored.merge(holiday_lookup(fold_val_raw), on=["unique_id", "ds"], how="left")
        pred = np.clip(scored[MODEL_ALIAS].to_numpy(), 0, None)
        scores.append(wmae(scored["y"], pred, scored["IsHoliday_eval"]))
    return scores

In [ ]:
with mlflow.start_run(run_name="DLinear_CrossValidation", nested=True):
    cv_scores = cv_dlinear(input_size=104, max_steps=500)
    for i, score in enumerate(cv_scores, start=1):
        mlflow.log_metric(f"fold_{i}_wmae", score)
    mlflow.log_metric("mean_cv_wmae", float(np.mean(cv_scores)))

## MLflow Stage: HPO

Keep DLinear tuning small. The useful knobs are mostly `input_size`, learning rate, and training steps. Increase only after the baseline run is stable.

In [ ]:
SEARCH_SPACE = [
    {"input_size": 52, "max_steps": 500},
    {"input_size": 104, "max_steps": 500},
    {"input_size": 104, "max_steps": 1000},
]

with mlflow.start_run(run_name="DLinear_HPO", nested=True):
    results = []
    for params in SEARCH_SPACE:
        scores = cv_dlinear(**params)
        results.append({**params, "mean_wmae": float(np.mean(scores))})
    hpo_results = pd.DataFrame(results).sort_values("mean_wmae")
    best_params = hpo_results.iloc[0].drop("mean_wmae").to_dict()
    hpo_results.to_csv(ROOT / "models" / "dlinear_hpo_results.csv", index=False)
    mlflow.log_params(best_params)
    mlflow.log_metric("best_mean_cv_wmae", float(hpo_results.iloc[0]["mean_wmae"]))
    mlflow.log_artifact(str(ROOT / "models" / "dlinear_hpo_results.csv"))

## MLflow Stage: Final Model

The final training uses all training rows and predicts the raw test horizon. A production-grade version should wrap `to_nixtla` and the fitted `NeuralForecast` object as `mlflow.pyfunc.PythonModel` so inference can start from raw test rows.

In [ ]:
with mlflow.start_run(run_name="DLinear_Final", nested=True):
    final_params = globals().get("best_params", {"input_size": 104, "max_steps": 500})
    nf_train = to_nixtla(train_raw)
    nf_test = to_nixtla(test_raw, history_frame=train_raw, include_y=False)
    final_nf = NeuralForecast(models=[build_dlinear(**final_params)], freq="W-FRI")
    final_nf.fit(df=nf_train)
    test_preds = final_nf.predict(futr_df=nf_test)
    test_pred_values = np.clip(test_preds[MODEL_ALIAS].to_numpy(), 0, None)
    make_submission(test_raw, test_pred_values, ROOT / "models" / "dlinear_submission.csv")
    mlflow.log_params(final_params)
    mlflow.log_artifact(str(ROOT / "models" / "dlinear_submission.csv"))
    # Optional after wrapper is finalized:
    # mlflow.pyfunc.log_model(artifact_path="model", python_model=DLinearRawPyFunc(...))

mlflow.end_run()

## README Takeaways

- DLinear is a fast deep-learning baseline that decomposes each series into linear trend and seasonal components.
- It is useful as a complexity check: if it is close to N-BEATS, the extra neural complexity may not be worth it.
- Compare DLinear against LightGBM especially on holiday weeks, because DLinear may smooth sharp holiday spikes.